# 🔮 Tarot Card Classifier — Training Notebook
EfficientNet-B0 fine-tuned on your tarot card photos.

**Before running:** Make sure Runtime > Change runtime type > **T4 GPU** is selected.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Set Your Data Path

In [ ]:
# Change this to where your data/raw folder is in Google Drive
DATA_DIR = '/content/drive/MyDrive/fortune-teller/data/raw'
SAVE_DIR = '/content/drive/MyDrive/fortune-teller/checkpoints'

import os
os.makedirs(SAVE_DIR, exist_ok=True)

# Check your card folders are visible
folders = sorted(os.listdir(DATA_DIR))
print(f'Found {len(folders)} card classes:')
for f in folders:
    count = len(os.listdir(os.path.join(DATA_DIR, f)))
    print(f'  {f}: {count} images')

## Step 3: Install Dependencies

In [ ]:
!pip install -q torch torchvision tqdm

## Step 4: Clone Your GitHub Repo

In [ ]:
!git clone https://github.com/Wannakorn-Sangthongngam/fortune-teller.git
%cd fortune-teller

## Step 5: Model Definition

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class TarotClassifier(nn.Module):
    def __init__(self, num_classes=78, dropout=0.3):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

    def freeze_backbone(self):
        for param in self.backbone.features.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.features.parameters():
            param.requires_grad = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 6: Dataset & DataLoaders

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from pathlib import Path

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class TarotDataset(Dataset):
    def __init__(self, root, transform):
        self.root = Path(root)
        self.transform = transform
        class_dirs = sorted([d for d in self.root.iterdir() if d.is_dir()])
        self.classes = [d.name for d in class_dirs]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = []
        for d in class_dirs:
            label = self.class_to_idx[d.name]
            for img_path in d.iterdir():
                if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        return self.transform(image), label

full_ds = TarotDataset(DATA_DIR, train_transform)
n = len(full_ds)
n_val   = int(n * 0.15)
n_test  = int(n * 0.10)
n_train = n - n_val - n_test

train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Classes: {len(full_ds.classes)}')

## Step 7: Training Loop

In [ ]:
import json
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

def accuracy(outputs, labels):
    return (outputs.argmax(dim=1) == labels).float().mean().item()

def run_epoch(model, loader, criterion, optimizer, train):
    model.train() if train else model.eval()
    total_loss, total_acc = 0.0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            total_acc  += accuracy(outputs, labels)
    n = len(loader)
    return total_loss / n, total_acc / n

# Save class mapping
with open(f'{SAVE_DIR}/classes.json', 'w') as f:
    json.dump(full_ds.classes, f, indent=2)

model = TarotClassifier(num_classes=len(full_ds.classes)).to(DEVICE)
model.freeze_backbone()
criterion = nn.CrossEntropyLoss()
best_val_acc = 0.0

# Phase 1
print('=== Phase 1: Train head only ===')
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

for epoch in range(1, 11):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer, train=True)
    vl, va = run_epoch(model, val_loader,   criterion, optimizer, train=False)
    scheduler.step()
    print(f'[P1 {epoch:02d}/10] loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pt')
        print(f'  Saved best model (val_acc={va:.3f})')

# Phase 2
print('\n=== Phase 2: Fine-tune full model ===')
model.unfreeze_backbone()
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

for epoch in range(1, 21):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer, train=True)
    vl, va = run_epoch(model, val_loader,   criterion, optimizer, train=False)
    scheduler.step()
    print(f'[P2 {epoch:02d}/20] loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pt')
        print(f'  Saved best model (val_acc={va:.3f})')

## Step 8: Test Evaluation

In [ ]:
model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pt'))
test_loss, test_acc = run_epoch(model, test_loader, criterion, None, train=False)
print(f'Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.3f}')

## Step 9: Test on a Single Image

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

def predict(image_path, top_k=3):
    image = PILImage.open(image_path).convert('RGB')
    tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    top_probs, top_idxs = probs.topk(top_k)
    print('Predictions:')
    for prob, idx in zip(top_probs, top_idxs):
        print(f'  {full_ds.classes[idx]:30s} {prob.item()*100:.1f}%')
    plt.imshow(image)
    plt.axis('off')
    plt.title(full_ds.classes[top_idxs[0]])
    plt.show()

# predict('/content/drive/MyDrive/fortune-teller/test_card.jpg')